# Resting-State Functional Connectivity Analysis

This notebook demonstrates a complete workflow for analyzing resting-state fMRI data:
1. Fetch open-access neuroimaging data
2. Extract regional time series using a brain atlas
3. Compute functional connectivity matrices
4. Visualize brain network organization
5. Analyze the Default Mode Network (DMN)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import datasets, plotting

# Import our custom modules
import sys
sys.path.insert(0, '..')
from src.connectivity import ConnectivityAnalyzer

sns.set_theme(style='white', font_scale=1.1)
%matplotlib inline

## 1. Initialize the Analyzer

We use the **Harvard-Oxford** atlas, which parcellates the brain into 48 cortical
and 21 subcortical regions — a good balance between granularity and interpretability.

In [ ]:
analyzer = ConnectivityAnalyzer(
    atlas='harvard-oxford',
    low_pass=0.1,
    high_pass=0.01,
)

## 2. Fetch Resting-State fMRI Data

Nilearn provides preprocessed developmental fMRI datasets.
We'll start with a single subject for demonstration.

In [ ]:
fmri_files = analyzer.fetch_data(n_subjects=1)
print(f'Downloaded: {fmri_files[0]}')

## 3. Extract Regional Time Series

The masker averages the BOLD signal within each atlas region,
producing one time series per brain region.

In [ ]:
time_series = analyzer.extract_time_series()

# Plot example time series from 5 regions
fig, ax = plt.subplots(figsize=(14, 5))
for i in range(5):
    ax.plot(time_series[:, i], label=analyzer.labels[i], alpha=0.8)
ax.set_xlabel('Time (volumes)')
ax.set_ylabel('BOLD Signal (z-scored)')
ax.set_title('Regional BOLD Time Series (5 example regions)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

## 4. Compute Functional Connectivity

Pearson correlation between every pair of regions gives us
the full connectivity matrix.

In [ ]:
conn_matrix = analyzer.compute_connectivity(method='correlation')
analyzer.summary(conn_matrix)

In [ ]:
# Visualize the full connectivity matrix
fig = analyzer.plot_matrix(
    conn_matrix,
    title='Resting-State Functional Connectivity (Pearson r)',
    save_path='../figures/connectivity_matrix_example.png'
)
plt.show()

## 5. Glass Brain Visualization

Project the strongest connections onto a transparent brain rendering.

In [ ]:
fig = analyzer.plot_glass_brain(
    conn_matrix,
    threshold=0.4,
)
plt.savefig('../figures/glass_brain_example.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Statistical Thresholding

Apply FDR correction to keep only statistically significant connections.

In [ ]:
thresholded = analyzer.threshold_connections(
    method='fdr',
    alpha=0.05
)

fig = analyzer.plot_matrix(
    thresholded,
    title='FDR-Corrected Connectivity (q < 0.05)',
)
plt.show()

## 7. Default Mode Network Analysis

The DMN is a set of brain regions active during rest and self-referential thought.
Let's extract and visualize its internal connectivity.

In [ ]:
dmn_matrix, dmn_labels, dmn_indices = analyzer.extract_dmn(conn_matrix)

if dmn_matrix is not None:
    print(f'DMN regions found: {len(dmn_labels)}')
    for label in dmn_labels:
        print(f'  - {label}')
    
    fig = analyzer.plot_matrix(
        dmn_matrix,
        labels=dmn_labels,
        title='Default Mode Network — Internal Connectivity',
        figsize=(8, 7),
        save_path='../figures/dmn_connectivity.png'
    )
    plt.show()

## 8. Compare Connectivity Methods

Partial correlation removes indirect connections,
revealing more direct functional relationships.

In [ ]:
partial_matrix = analyzer.compute_connectivity(method='partial_correlation')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(conn_matrix, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, ax=axes[0], cbar_kws={'shrink': 0.7})
axes[0].set_title('Pearson Correlation', fontsize=13, fontweight='bold')

sns.heatmap(partial_matrix, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
            square=True, ax=axes[1], cbar_kws={'shrink': 0.7})
axes[1].set_title('Partial Correlation (Ledoit-Wolf)', fontsize=13, fontweight='bold')

plt.suptitle('Full vs Partial Correlation Connectivity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Summary

This analysis demonstrated:
- Atlas-based parcellation of resting-state fMRI data
- Full and partial correlation connectivity estimation
- FDR-corrected statistical thresholding
- Targeted DMN subnetwork extraction

**Next steps**: Group-level analysis across multiple subjects, graph-theoretic
network metrics, and cross-modal integration with EEG source localization.